# Template 04c: Feature Encoding

**Inputs:**
- data/04_train.parquet
- data/04_test.parquet
- config_generated/master_feature_encoding.csv

**Outputs:**
- data/04c_train_encoded.parquet
- data/04c_test_encoded.parquet
- models/04c_encoders.pkl (for holdout)
- results/04c_encoding_summary.csv

In [ ]:
config_path = "config/car_coll/v1"

In [ ]:
import pandas as pd
import yaml
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment
from feature_encoder import apply_master_encoding, save_encoders

print("########################################")
print("# STAGE 04c: FEATURE ENCODING")
print("########################################")

project_root = setup_notebook_environment()

In [ ]:
config_file = f'{config_path}/config.yaml'
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

output_base = cfg['paths']['output_base']
print(f'Output: {output_base}')

In [ ]:
# Load train/test splits
train_file = f'{output_base}/data/04_train.parquet'
test_file = f'{output_base}/data/04_test.parquet'

print(f'\n* Loading data...')
train = pd.read_parquet(train_file)
test = pd.read_parquet(test_file)

print(f'  Train: {train.shape}')
print(f'  Test: {test.shape}')

In [ ]:
# Apply master encoding
print(f'\n* Applying master feature encoding...')

train_encoded, test_encoded, encoders, encoding_summary = apply_master_encoding(
    train, test, config_path,
    min_frequency=0.01,  # 1% threshold
    min_count=50         # OR 50 observations
)

In [ ]:
# Display encoding summary
print(f'\n* Encoding Summary:')
print(f'\nEncoding types used:')
print(encoding_summary['encoding_type'].value_counts())

print(f'\nCategories with __OTHER__:')
print(encoding_summary[encoding_summary['has_other']== True][['original_column', 'n_categories']].head(10))

print(f'\nCategories with __MISSING__:')
print(encoding_summary[encoding_summary['has_missing'] == True][['original_column', 'n_categories']].head(10))

In [ ]:
# Save encoded data
train_output = f'{output_base}/data/04c_train_encoded.parquet'
test_output = f'{output_base}/data/04c_test_encoded.parquet'

train_encoded.to_parquet(train_output, index=False)
test_encoded.to_parquet(test_output, index=False)

print(f'\n* Saved:')
print(f'  {train_output}')
print(f'  {test_output}')

In [ ]:
# Save encoders for holdout
save_encoders(encoders, encoding_summary, output_base)

In [ ]:
print("\n########################################")
print("# STAGE 04c: COMPLETE")
print("########################################")